# Linear regression
This notebook covers one of the most beginner-friendly ML algorithms: _Least Squares_, which is basic for a linear regression to be performed.

## Step 1: Import required libraries
This step can be skipped, but it's strongly recommended, since the dependencies used to build this project are downloaded.

To do so, a bash script is used so as to install basic dependencies (Makefile, g++...) as well as the ones that are meant to be installed via _pip_ (such as **NumPy** and **Pandas**).

In [ ]:
%%bash
/$(pwd)/deps/deps.sh

## Step 2: Import functions from modules
After dependency files are already imported, functions to be used afterwards must be imported. For such task to be fulfilled, path to custom libraries should be added first. Use **_sys.path.append_** for this purpose so that Python's module tracker is able to find them.

In [ ]:
# Whenever a .py file is modified, changes are automatically reflected in the notebook.
%load_ext autoreload
%autoreload 2

# Add user's source files directory to Python's module search path.
import sys
from pathlib import Path
sys.path.append(
    str(Path("./python3/src").resolve())
)

# Import libraries (both built-in and custom).
import pandas as pd
from synthetic_data_generation import generateLinearData as genLinData
from data_preprocessing import preprocessData
from linear_regression import fitLinearData1D
from data_logger import DataLogger as DLogger
from plotting import plotLinePlot

## Step 3: Generate linear data
Once dependencies are sorted out, data generation can start. In this case, some noisy linear data will be created. It will be describable by the equation below:

$$
f(x) = 3x + 7 + \mathcal{N}(0, 10^2)
$$

To do so, **generateLinearData** function found within _synthetic_data_generation_ module will be used. Its output is a Pandas DataFrame. Default values for the mentioned function will lead to the function following the formula above to be generated (100 data points).

In [ ]:
lin_data: pd.DataFrame = genLinData()
plotLinePlot(lin_data, x_axis_label = "X axis", y_axis_label = "F(x)", save_plot = False, display_plot = True, plot_name = "Noisy linear data")

## Step 4: Preprocess noisy data
Generated data is noisy, so it includes some points that should not be taken into account by the time linear regression is being performed. These type of points can be either:
 - Non existing points: if for a given row, the value associated to any of its column is _Nan_, then the whole row needs to be erasesd. It may happen that the data for X exists but not for Y and vice-versa.
 - Outliers: _**3σ rule**_ is applied, an empirical rule which assumes that 99.7% of the data falls within three standard deviations (https://en.wikipedia.org/wiki/68%E2%80%9395%E2%80%9399.7_rule).

In [ ]:
cl_lin_data: pd.DataFrame = preprocessData(lin_data)
plotLinePlot(cl_lin_data, x_axis_label = "X axis", y_axis_label = "F(x)", save_plot = False, display_plot = True, plot_name = "Clean linear data")

## Step 5: perform linear regression
Once synthetic linear data is clean, it's time to perform linear regression over provided data. *__Least Squares Method__* (LSM) is going to be used for such purpose, which is explained in a pretty intuitive way.

Some data points are provided:

$$
f(x) = (x_1, y_1), (x_1, y_1), ..., (x_n, y_n)
$$

And it is assumed that:

$$
f(x) ≈ mx + b
$$

So the question is: _what values of **m** and **b** fit best the data in question?_

LSM's fundamental criterion is to pick the line that makes the total squared vertical error as small as possible. For each point, the error can be defined as follows:

$$
e_i = y_i - (mx_i + b)
$$

Therefore, the total error can be calculated by summing all of the errors:

$$
L(m,b) = \sum_{i=1}^{n} e_i^2 = \sum_{i=1}^{n} (y_i - (mx_i + b_i))^2
$$

One might ask why squared error is being used. Well, there are three main reasons for that:
 - Cancellations are avoided: positive and negative errors would cancel each other if using raw error.
 - It penalizes big errors more (since they are squared).
 - It makes the math solvable (noticeable when derivatives turn up later).

So far, the total error function (**_L(m, b)_**) has been defined. Now, the aim is to calculate the values for **m** and **b** that minimize the total error. Since two variables are involved, partial derivatives should be taken and equaled to zero for both **m** and **b**:

$$
\frac{\partial L}{\partial m} = 0
$$

$$
\frac{\partial L}{\partial b} = 0
$$

For **m**:

$$
\frac{\partial L}{\partial m} = -2\sum_{i=1}^{n} x_i(y_i - (mx_i + b_i)) = 0 \Rightarrow \sum_{i=1}^{n} x_iy_i = m\sum_{i=1}^{n} x_i^2 + b\sum_{i=1}^{n} x_i
$$

For **b**:
$$
\frac{\partial L}{\partial b} = -2\sum_{i=1}^{n} (y_i - mx_i - b) = 0 \Rightarrow \sum_{i=1}^{n} y_i = m\sum_{i=1}^{n} (x_i + nb)
$$

The equations system above can be solved for **b** and **m** (intermediate steps are omitted):

$$
m = \frac{\sum_{i=1}^{n} ((x_i - \bar{x})(y_i - \bar{y}))}{\sum_{i=1}^{n} (x_i - \bar{x})}
$$

$$
b = \bar{y} - m\bar{x}
$$

All of the steps above are comprised in a function called _**fitLinearData1D**_ that has already been defined within _**linear_regression**_ module.

In [ ]:
b, m = fitLinearData1D(cl_lin_data)  # f(x) = m·x + b
plotLinePlot(cl_lin_data, intercept = b, slope = m, x_axis_label = "X axis", y_axis_label = "F(x)", save_plot = False, display_plot = True, plot_name = "Linear regression")